# Preprocessing Hepatitis Data

### Import libraries

In [27]:
import pandas as pd
import sklearn.preprocessing as sp
from sklearn.model_selection import train_test_split
import numpy as np
import warnings

warnings.filterwarnings(action='ignore', category=FutureWarning)

### Import data


In [28]:
columns = [
    "Class",
    "Age",
    "Sex",
    "Steroid",
    "Antivirals",
    "Fatigue",
    "Malaise",
    "Anorexia",
    "Liver Big",
    "Liver Firm",
    "Spleen Palpable",
    "Spiders",
    "Ascites",
    "Varices",
    "Bilirubin",
    "Alk Phosphate",
    "Sgot",
    "Albumin",
    "Protime",
    "Histology",
]
data = pd.read_csv("../raw/hepatitis/hepatitis.data", header=None, names=columns)
print(data.shape)

(155, 20)


In [29]:
data.replace("?", np.nan, inplace=True)

### Separate train and test

In [30]:
# separte train and test data

X = data.drop("Class", axis=1)
# Raw Class labels are 1 = death and 2 = survival; encode death as the bad event (1).
y = data["Class"].map({1: 1, 2: 0}).rename("target")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Missing Values

In [31]:
categorical_columns = [
    "Sex",
    "Steroid",
    "Antivirals",
    "Fatigue",
    "Malaise",
    "Anorexia",
    "Liver Big",
    "Liver Firm",
    "Spleen Palpable",
    "Spiders",
    "Ascites",
    "Varices",
    "Histology",
]
numerical_columns = [
    "Age",
    "Bilirubin",
    "Alk Phosphate",
    "Sgot",
    "Albumin",
    "Protime",
]

In [32]:
# convert numeric columns
for col in numerical_columns:
    X_train[col] = pd.to_numeric(X_train[col])
    X_test[col] = pd.to_numeric(X_test[col])

In [33]:
# Fill in missing values with the most frequent value in each column for categorical columns seprately for train and test data

for column in categorical_columns:
    most_frequent_value = X_train[column].mode()[0]
    X_train[column].fillna(most_frequent_value, inplace=True)
    X_test[column].fillna(most_frequent_value, inplace=True)

# Fill in missing values with the mean value in each column for numerical columns
for column in numerical_columns:
    mean_value = X_train[column].mean()
    X_train[column].fillna(mean_value, inplace=True)
    X_test[column].fillna(mean_value, inplace=True)

### Normalize numerical columns

In [34]:
# scale values of numerical columns 
scaler = sp.StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [35]:
# The target was already encoded before splitting: 1 = death, 0 = survival.
y_train = y_train.astype(int)
y_test = y_test.astype(int)

### Export data

In [36]:
#print target proportions
print("Train target proportions:")
print(y_train.value_counts(normalize=True))

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True))

Train target proportions:
Class
1    0.790323
0    0.209677
Name: proportion, dtype: float64

Test target proportions:
Class
1    0.806452
0    0.193548
Name: proportion, dtype: float64


In [37]:
train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)
disease_name = "hepatitis"

In [38]:
from pathlib import Path

output_dir = Path("../pre-processed")  # adjust if your notebook runs elsewhere
output_dir.mkdir(parents=True, exist_ok=True)

train_data.to_csv(output_dir / f"{disease_name}_train.csv", index=False)
test_data.to_csv(output_dir / f"{disease_name}_test.csv", index=False)